# 05 · Autos vs niebla: distancia de detección

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tomas-pucv/Camanchaca-Predict/blob/main/notebooks/05_cars_vs_fog.ipynb)

**Propósito.** Enfoque de "objeto a distancia conocida": recall de un detector COCO (Faster R-CNN
v2 de torchvision) sobre los autos de Cityscapes *val* en función de su distancia real (estéreo),
con y sin niebla (Foggy Cityscapes β = 0,005/0,01/0,02 y CS-fog V = 50/100 m). Resume la distancia
a la que la niebla impide detectar autos (`V_det`) y la compara con el MOR.

**Tiempo esperado (T4).** ~10 min (2048×1024, AMP).
**Salidas.** `runs/cars/detections.csv`, `reports/tables/cars_recall.{csv,md}`,
`reports/tables/cars_vdet.md`, `reports/figures/cars_recall_vs_distance.png`.

## Configuración común
Misma celda en todos los notebooks (detalles en `00_setup_colab`). En **Colab**: monta Drive,
fija `CAMANCHACA_DATA=/content/data` y `CAMANCHACA_RUNS=/content/drive/MyDrive/camanchaca/runs`,
clona/actualiza el repo en la referencia `REF`, instala el paquete (`pip install -e ".[demo,data]"`,
**sin** reinstalar torch) y desempaqueta los `.tar` de datos de Drive (salvo `NEEDS_DATA = False`). **Fuera de Colab** (`SMOKE=True`)
usa datos sintéticos y pesos aleatorios generados por `camanchaca smoke`: sirve para verificar que
todo corre, **sus números no son resultados**.

In [ ]:
# Configuración común (idéntica en los notebooks 00-06)
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = not IN_COLAB   # True: datos sintéticos + pesos aleatorios (solo verifica que todo corre)
REF = "main"           # rama/tag/commit del repo; para la entrega final usar "v1.0-final"
REPO_URL = "https://github.com/Tomas-pucv/Camanchaca-Predict.git"
DRIVE_DIR = Path("/content/drive/MyDrive/camanchaca")   # tars de datos, corridas, pesos y credenciales

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["CAMANCHACA_DATA"] = "/content/data"
    os.environ["CAMANCHACA_RUNS"] = str(DRIVE_DIR / "runs")
    REPO = Path("/content/Camanchaca-Predict")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
    !git -C {REPO} fetch -q --all --tags && git -C {REPO} checkout -q {REF} && (git -C {REPO} pull -q --ff-only 2>/dev/null || true)
    %cd {REPO}
    !pip install -q -e ".[demo,data]"   # NO reinstala torch/torchvision: se usan los de Colab
    if not SMOKE and globals().get("NEEDS_DATA", True):
        for tar in sorted((DRIVE_DIR / "data").glob("*.tar")):   # imágenes preprocesadas (notebook 01)
            !camanchaca unpack --tar {tar}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    %cd {REPO}
    if shutil.which("camanchaca") is None:
        print("AVISO: falta el comando `camanchaca`: ejecutar  pip install -e '.[dev,demo]'")

WORK = Path(os.environ.get("CAMANCHACA_SMOKE_DIR", str(REPO / "local_workspace" / "nb_smoke")))
if SMOKE:
    os.environ.update(CAMANCHACA_DATA=str(WORK / "data"), CAMANCHACA_RUNS=str(WORK / "runs"),
                      CAMANCHACA_REPORTS=str(WORK / "reports"), HF_HUB_OFFLINE="1")
    if not (WORK / "smoke.yaml").exists():   # ~4-6 min en CPU, una sola vez
        !camanchaca smoke --workdir {WORK}

sys.path.insert(0, str(REPO / "src"))   # importable sin reiniciar el kernel tras `pip install -e`
from camanchaca.config import load_yaml
from camanchaca.paths import data_root, reports_dir, runs_root

EXP = str(WORK / "smoke.yaml") if SMOKE else "configs/experiment/main.yaml"
EXP_NAME = load_yaml(EXP)["name"]
print(f"Colab={IN_COLAB} | SMOKE={SMOKE} | EXP={EXP}")
print(f"datos={data_root()} | corridas={runs_root()} | reportes={reports_dir()}")

In [ ]:
MANIFESTS = WORK / "manifests" if SMOKE else REPO / "data" / "manifests"
DEVICE = "cuda" if IN_COLAB else "cpu"
EXTRA = "--weights none --bootstrap 50" if SMOKE else ""   # humo: detector con pesos aleatorios, sin descarga
!camanchaca cars --images {MANIFESTS}/cars_images.csv --gt {MANIFESTS}/cars_gt.csv --device {DEVICE} --score-thr 0.5 --iou 0.5 {EXTRA}

In [ ]:
from IPython.display import Image, Markdown, display
TABLES, FIGS = reports_dir() / "tables", reports_dir() / "figures"
for name in ["cars_recall.md", "cars_vdet.md"]:
    if (TABLES / name).exists(): display(Markdown((TABLES / name).read_text()))
if (FIGS / "cars_recall_vs_distance.png").exists(): display(Image(filename=str(FIGS / "cars_recall_vs_distance.png"), width=800))

## Cómo leerlo (censura)
- La distancia de cada auto es la **mediana estéreo** sobre su máscara; más allá de ~100–150 m el
  estéreo de Cityscapes es ruidoso y hay pocos autos: los últimos intervalos tienen IC anchos.
- Los autos lejanos son pequeños: el recall cae con la distancia **aun sin niebla**. Comparar
  siempre contra la condición `clear` (misma imagen) para separar tamaño de niebla.
- `V_det` está **censurado**: si el recall no cae bajo el umbral dentro del rango medido, solo se
  sabe que `V_det` es mayor que la última distancia observada (se reporta como "≥").
- Se espera `V_det` < MOR: detectar un auto exige más contraste que el 5 % del MOR.
- Los autos truncados o sin profundidad válida se excluyen (`truncated`, `depth_ok` en `cars_gt.csv`).